# 🛡️ Risk Report – Quant Analyst
**Workflow**: Multi-Asset Returns → Correlation Heatmap → VaR & CVaR → Tail Risk Distribution

---

In [1]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

session = AnalystSession(role=RoleContext.ANALYST)

SYMBOLS = ['BTC-USD', 'ETH-USD', 'SOL-USD']
DAYS = 365
CONFIDENCE = 0.95

PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

## 1. Portfolio Data Prep
Aggregating returns for a diversified crypto portfolio.

In [2]:
returns_data = []
for sym in SYMBOLS:
    try:
        df_asset = session.load_ohlcv(sym, '1d')
    except Exception:
        df_asset = session.sample_ohlcv(symbol=sym, days=DAYS)
    df_asset = session.make_returns(df_asset)
    returns_data.append(df_asset.select(['timestamp', 'returns']).rename({'returns': sym}))

# Join and convert to numpy
merged = returns_data[0]
for other in returns_data[1:]:
    merged = merged.join(other, on='timestamp', how='inner')

merged = merged.drop_nulls()
R = merged.select(SYMBOLS).to_numpy()
print(f"Joined Returns shape: {R.shape}")

SchemaError: datatypes of join keys don't match - `timestamp`: datetime[μs] on left does not match `timestamp`: datetime[μs, Asia/Ho_Chi_Minh] on right (and no other type was available to cast to)

## 2. Dynamic Correlation Heatmap
Visualizing inter-asset dependencies via Plotly.

In [ ]:
corr = np.corrcoef(R.T)
fig_corr = px.imshow(
    corr, x=SYMBOLS, y=SYMBOLS, 
    color_continuous_scale='RdBu_r', 
    zmin=-1, zmax=1,
    title="Portfolio Asset Correlation"
)
fig_corr.update_layout(**PLOTLY_DARK)
fig_corr.show()

## 3. VaR & CVaR Analysis
Quantifying the expected loss in the worst-case scenario.

In [ ]:
w = np.ones(len(SYMBOLS)) / len(SYMBOLS)
port_returns = R @ w

var_val = np.percentile(port_returns, (1 - CONFIDENCE) * 100)
cvar_val = port_returns[port_returns <= var_val].mean()

fig_dist = px.histogram(port_returns, nbins=100, title="Portfolio Return Distribution & Tail Risk")
fig_dist.add_vline(x=var_val, line_dash="dash", line_color="orange", annotation_text=f"VaR {CONFIDENCE*100:.0f}%")
fig_dist.add_vline(x=cvar_val, line_dash="dot", line_color="red", annotation_text=f"CVaR {CONFIDENCE*100:.0f}%")
fig_dist.update_layout(**PLOTLY_DARK)
fig_dist.show()

print(f"95% VaR: {var_val:.4f} | 95% CVaR: {cvar_val:.4f}")